# Bab 7 — Clustering Workflow

Notebook pendamping Bab 7 Level 3.

Notebook ini mengikuti halaman bab di situs. Jalankan sel dari atas ke bawah. Penjelasan tiap baris kode ada di halaman bab.

In [ ]:
import os
from pathlib import Path

# Pindah ke folder utama proyek agar "data/mall_customers.csv" ketemu.
if not Path("data").exists():
    os.chdir("../..")

## Persiapan: pustaka yang dipakai

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.preprocessing import StandardScaler

## Langkah 1: Memahami dataset

In [ ]:
df = pd.read_csv("data/mall_customers.csv")

print("Ukuran:", df.shape)
print("Nilai kosong total:", int(df.isna().sum().sum()))
print("Baris duplikat:", int(df.duplicated().sum()))
print(df.describe().round(1).loc[["min", "mean", "max"]])

## Langkah 2: Menyiapkan fitur

In [ ]:
features = ["Annual Income (k$)", "Spending Score (1-100)"]
X = df[features]
print("Fitur untuk clustering:", features)
print("Kolom yang tidak dipakai:", [c for c in df.columns if c not in features])

## Langkah 3: Menyamakan skala

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print("Rata-rata setelah skala:", (X_scaled.mean(axis=0).round(2) + 0.0))
print("Simpangan baku setelah skala:", X_scaled.std(axis=0).round(2))

## Langkah 4: Memilih jumlah kelompok

In [ ]:
scores = {}
for k in range(2, 9):
    labels_k = KMeans(n_clusters=k, n_init=10, random_state=42).fit_predict(X_scaled)
    scores[k] = silhouette_score(X_scaled, labels_k)

for k, s in scores.items():
    print(f"k = {k}: silhouette = {s:.3f}")
best_k = max(scores, key=scores.get)
print("k dengan silhouette tertinggi:", best_k)

fig, ax = plt.subplots(figsize=(5.5, 4))
ax.plot(list(scores), list(scores.values()), marker="o", color="#0072B2")
ax.set_xlabel("Jumlah kelompok (k)")
ax.set_ylabel("Silhouette score")
ax.set_title("Memilih jumlah kelompok")
plt.show()

## Langkah 5: Melatih model dan memberi nomor kelompok

In [ ]:
model = KMeans(n_clusters=best_k, n_init=10, random_state=42)
df["Cluster"] = model.fit_predict(X_scaled)
print(df["Cluster"].value_counts().sort_index())

## Langkah 6: Memeriksa kestabilan

In [ ]:
for seed in range(5):
    other = KMeans(n_clusters=best_k, n_init=10, random_state=seed).fit_predict(X_scaled)
    ari = adjusted_rand_score(df["Cluster"], other)
    print(f"random_state = {seed}: kesamaan dengan hasil utama (ARI) = {ari:.3f}")

## Langkah 7: Menafsirkan kelompok

In [ ]:
profile = df.groupby("Cluster")[features + ["Age"]].mean().round(1)
profile.insert(0, "Jumlah", df["Cluster"].value_counts().sort_index())
print(profile)

## Langkah 8: Visualisasi akhir

In [ ]:
colors = ["#0072B2", "#E69F00", "#009E73", "#CC79A7", "#D55E00", "#56B4E9", "#999999", "#000000"]
centers_original = scaler.inverse_transform(model.cluster_centers_)

fig, ax = plt.subplots(figsize=(6, 4.8))
for k in range(best_k):
    sub = df[df["Cluster"] == k]
    ax.scatter(sub[features[0]], sub[features[1]], s=22, color=colors[k], label=f"Kelompok {k}")
ax.scatter(centers_original[:, 0], centers_original[:, 1], marker="X", s=140, color="black", label="Centroid")
ax.set_xlabel(features[0])
ax.set_ylabel(features[1])
ax.set_title("Segmentasi pelanggan Mall Customers")
ax.legend(fontsize=8)
plt.show()

## Langkah 9: Laporan ringkas

In [ ]:
for k, row in profile.iterrows():
    share = row["Jumlah"] / len(df) * 100
    print(
        f"Kelompok {k}: {int(row['Jumlah'])} pelanggan ({share:.1f}%), "
        f"pendapatan rata-rata {row[features[0]]}, skor belanja rata-rata {row[features[1]]}, "
        f"usia rata-rata {row['Age']}"
    )

## Latihan

Kerjakan bagian **Hands-on Exercises** pada halaman bab. Jawabannya ada di bagian yang dapat dibuka pada halaman tersebut.